# Project 05 — Colorectal Cancer Atlas: Harmony vs scVI

**Dataset:** Pelka et al. 2021, *Cell* — "Spatially organized multicellular immune hubs in human colorectal cancer"
DOI: 10.1016/j.cell.2021.08.003 | CELLxGENE: collection `4d82bd8e-8827-410b-8fc7-685b7dd92585`

**Goal:** Side-by-side benchmark of two batch-correction strategies on a multi-patient CRC tumor cohort.

**Subset:** 4 MMRd (MSI-like) + 4 MMRp (MSS-like) patients, tumor samples only (~40–50k cells).

**Pipeline:**
1. QC + normalize
2. Harmony integration (patient-level batch correction)
3. scVI integration (deep generative model, same batch key)
4. Leiden clustering on both embeddings
5. Lineage validation against author labels (`ClusterTop`)
6. Integration quality metric (mixing entropy)
7. MMRd vs MMRp compositional analysis


## 1. Setup

In [ ]:
import os, sys, gc, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

PROJECT_DIR = Path('/home/marko-b2/upwork_portfolio/05_colorectal_atlas')
DATA_DIR = PROJECT_DIR / 'data'
FIG_DIR  = PROJECT_DIR / 'figures'
RES_DIR  = PROJECT_DIR / 'results'
FIG_DIR.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)

# Apply portfolio style if present
style_path = Path('/home/marko-b2/upwork_portfolio/_shared/styles/portfolio.mplstyle')
if style_path.exists():
    plt.style.use(str(style_path))

print('scanpy:', sc.__version__)
print('anndata:', ad.__version__)


## 2. Load Pelka 2021 and select 4 MMRd + 4 MMRp patients

The full atlas is 370k cells. We subset to 8 tumor donors (top by cell count within each MMR group),
restricted to `_T*` samples (primary tumor, excluding paired normals).

In [ ]:
H5AD = DATA_DIR / 'pelka2021_all_cells.h5ad'
adata_full = sc.read_h5ad(H5AD)
print('Full atlas:', adata_full.shape)
print('Disease:', dict(adata_full.obs['disease'].value_counts()))


In [ ]:
# Restrict to tumor samples only (PatientTypeID ends with _T, _TA, _TB, etc.)
tumor_mask = adata_full.obs['disease'] == 'colon adenocarcinoma'
print('Tumor cells (any MMR):', tumor_mask.sum())

# Per-donor cell counts within each MMR group (tumor only)
tumor = adata_full[tumor_mask].obs[['donor_id', 'MMRStatusTumor']].copy()
donor_mmr = (tumor.groupby('donor_id', observed=True)['MMRStatusTumor']
                  .agg(lambda s: s.mode()[0] if len(s.mode()) else 'unknown'))
donor_counts = tumor.groupby('donor_id', observed=True).size().rename('n_cells')
donor_summary = pd.concat([donor_mmr.rename('mmr'), donor_counts], axis=1)
donor_summary = donor_summary[donor_summary['mmr'].isin(['MMRd', 'MMRp'])]
donor_summary = donor_summary.sort_values(['mmr', 'n_cells'], ascending=[True, False])
print(donor_summary.groupby('mmr').head(6))


In [ ]:
# Pick top 4 per group
mmrd_donors = donor_summary[donor_summary['mmr'] == 'MMRd'].head(4).index.tolist()
mmrp_donors = donor_summary[donor_summary['mmr'] == 'MMRp'].head(4).index.tolist()
selected = mmrd_donors + mmrp_donors
print('Selected donors (MMRd):', mmrd_donors)
print('Selected donors (MMRp):', mmrp_donors)

mask = (adata_full.obs['donor_id'].isin(selected)) & tumor_mask
adata = adata_full[mask].copy()
del adata_full
gc.collect()
print('Subset shape:', adata.shape)
print('Cells per donor:')
print(adata.obs['donor_id'].value_counts())


## 3. QC

The CELLxGENE version of Pelka 2021 stores normalized expression in `.X` and may include raw counts
in `.raw`. We check and route counts appropriately for scVI (which needs raw counts).

In [ ]:
print('X dtype:', adata.X.dtype, '| min/max sample:',
      float(adata.X[:1000, :1000].min()),
      float(adata.X[:1000, :1000].max()))
if adata.raw is not None:
    print('raw shape:', adata.raw.shape)
    print('raw X dtype:', adata.raw.X.dtype)
    sample = adata.raw.X[:1000, :1000]
    print('raw min/max:', float(sample.min()), float(sample.max()))


In [ ]:
# Pelka in CELLxGENE: .X = log-normalized, .raw.X = raw counts (CELLxGENE convention)
# Make a working AnnData where layers['counts'] = raw counts (for scVI), .X = log-norm (for Harmony)
if adata.raw is not None:
    # Use raw genes to keep counts matrix
    counts = adata.raw.X.copy()
    if hasattr(counts, 'toarray'):
        pass  # keep sparse
    # Align genes: .raw has all genes, .X may be filtered. We use .raw's var.
    adata_work = ad.AnnData(
        X=adata.raw.X.copy(),
        obs=adata.obs.copy(),
        var=adata.raw.var.copy(),
    )
    adata_work.layers['counts'] = adata_work.X.copy()
else:
    adata_work = adata.copy()
    adata_work.layers['counts'] = adata_work.X.copy()

print('Working AnnData:', adata_work.shape)
print('Counts sample max (should be integer-like for raw):',
      float(adata_work.layers['counts'][:1000, :1000].max()))


In [ ]:
# Standard QC
adata_work.var['mt'] = adata_work.var_names.str.startswith('MT-') | adata_work.var_names.str.startswith('mt-')
# Many CELLxGENE objects use Ensembl IDs in var_names; fall back to feature_name if present
if adata_work.var['mt'].sum() == 0 and 'feature_name' in adata_work.var.columns:
    adata_work.var['mt'] = adata_work.var['feature_name'].astype(str).str.startswith('MT-')

sc.pp.calculate_qc_metrics(adata_work, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
print('Median n_genes:', float(adata_work.obs['n_genes_by_counts'].median()))
print('Median total counts:', float(adata_work.obs['total_counts'].median()))
print('Median pct_mt:', float(adata_work.obs['pct_counts_mt'].median()))


In [ ]:
# Light QC filter (Pelka data is already QC'd by authors, so this is a sanity pass)
n0 = adata_work.n_obs
adata_work = adata_work[adata_work.obs['n_genes_by_counts'] >= 500].copy()
adata_work = adata_work[adata_work.obs['pct_counts_mt'] <= 25].copy()
sc.pp.filter_genes(adata_work, min_cells=10)
print(f'After QC: {adata_work.shape} (dropped {n0 - adata_work.n_obs} cells)')


## 4. Normalize, log, HVG selection (batch-aware)

In [ ]:
# Normalize from raw counts -> log1p
adata_work.X = adata_work.layers['counts'].copy()
sc.pp.normalize_total(adata_work, target_sum=1e4)
sc.pp.log1p(adata_work)
adata_work.layers['lognorm'] = adata_work.X.copy()

# Batch-aware HVG selection (per donor)
sc.pp.highly_variable_genes(
    adata_work,
    n_top_genes=3000,
    flavor='seurat_v3',
    layer='counts',
    batch_key='donor_id',
)
print('HVGs selected:', int(adata_work.var['highly_variable'].sum()))


## 5. Baseline (no integration): PCA + UMAP

In [ ]:
# Snapshot full lognorm matrix for plotting and DEG (kept in .raw)
adata_work.raw = adata_work

# Subset to HVGs for scale + PCA (older Scanpy API: no mask_var arg on scale)
adata_hvg = adata_work[:, adata_work.var['highly_variable']].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=50)

# Carry PCA back to the main object
adata_work.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata_work.uns['pca'] = adata_hvg.uns.get('pca', {})
del adata_hvg

sc.pp.neighbors(adata_work, n_neighbors=15, n_pcs=30, use_rep='X_pca', key_added='nbrs_uncorr')
sc.tl.umap(adata_work, neighbors_key='nbrs_uncorr')
adata_work.obsm['X_umap_uncorr'] = adata_work.obsm['X_umap'].copy()
print('Baseline UMAP done. PCA shape:', adata_work.obsm['X_pca'].shape)


In [ ]:
# Quick look — donor effect should be visible
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.embedding(adata_work, basis='X_umap_uncorr', color='donor_id',
                ax=axes[0], show=False, title='Donor (uncorrected)',
                legend_loc='on data', legend_fontsize=6, frameon=False)
sc.pl.embedding(adata_work, basis='X_umap_uncorr', color='ClusterTop',
                ax=axes[1], show=False, title='Lineage (uncorrected)', frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_uncorrected.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Harmony integration (batch = donor_id)

In [ ]:
import harmonypy as hm

# Run harmonypy directly (skip scanpy.external wrapper to control orientation ourselves)
# harmonypy in newer versions returns Z_corr in (cells, PCs) — we detect and align.
ho = hm.run_harmony(
    adata_work.obsm['X_pca'].astype('float64'),
    adata_work.obs,
    'donor_id',
    max_iter_harmony=20,
)

Z = ho.Z_corr
print('Raw Z_corr shape from harmonypy:', Z.shape, '(n_obs =', adata_work.n_obs, ')')
# Align to (cells, PCs)
if Z.shape[0] == adata_work.n_obs:
    adata_work.obsm['X_pca_harmony'] = Z
elif Z.shape[1] == adata_work.n_obs:
    adata_work.obsm['X_pca_harmony'] = Z.T
else:
    raise RuntimeError(f'Unexpected Z_corr shape {Z.shape} vs n_obs {adata_work.n_obs}')
print('Stored X_pca_harmony shape:', adata_work.obsm['X_pca_harmony'].shape)

sc.pp.neighbors(adata_work, n_neighbors=15, n_pcs=30,
                use_rep='X_pca_harmony', key_added='nbrs_harmony')
sc.tl.umap(adata_work, neighbors_key='nbrs_harmony')
adata_work.obsm['X_umap_harmony'] = adata_work.obsm['X_umap'].copy()
sc.tl.leiden(adata_work, neighbors_key='nbrs_harmony', resolution=0.6,
             key_added='leiden_harmony', flavor='igraph', n_iterations=2, directed=False)
print('Harmony Leiden clusters:', adata_work.obs['leiden_harmony'].nunique())

## 7. scVI integration (batch = donor_id)

We train a VAE on raw counts with `donor_id` as the batch covariate. Settings chosen for CPU
throughput on a ~40k-cell subset: `n_latent=10`, `n_layers=2`, `n_hidden=128`, ~50 epochs.

In [ ]:
import torch
import scvi

print('scVI:', scvi.__version__, '| torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
torch.set_num_threads(8)  # respect 16-core machine, leave headroom

# scVI needs raw counts -- ensure we hand them the counts layer
adata_scvi = adata_work.copy()
# Subset to HVGs to speed up training
adata_scvi = adata_scvi[:, adata_scvi.var['highly_variable']].copy()
adata_scvi.X = adata_scvi.layers['counts'].copy()
print('scVI input:', adata_scvi.shape)

scvi.model.SCVI.setup_anndata(adata_scvi, layer='counts', batch_key='donor_id')

model = scvi.model.SCVI(
    adata_scvi,
    n_latent=10,
    n_layers=2,
    n_hidden=128,
    gene_likelihood='nb',
)
model.train(
    max_epochs=50,
    accelerator='cpu',
    batch_size=512,
    early_stopping=True,
    early_stopping_patience=8,
    check_val_every_n_epoch=1,
    plan_kwargs={'lr': 1e-3},
)


In [ ]:
# Pull latent embedding back into the main object
latent = model.get_latent_representation()
adata_work.obsm['X_scvi'] = latent
print('scVI latent shape:', latent.shape)

# Save trained model
model.save(str(RES_DIR / 'scvi_model'), overwrite=True)

sc.pp.neighbors(adata_work, n_neighbors=15, use_rep='X_scvi', key_added='nbrs_scvi')
sc.tl.umap(adata_work, neighbors_key='nbrs_scvi')
adata_work.obsm['X_umap_scvi'] = adata_work.obsm['X_umap'].copy()
sc.tl.leiden(adata_work, neighbors_key='nbrs_scvi', resolution=0.6,
             key_added='leiden_scvi', flavor='igraph', n_iterations=2, directed=False)
print('scVI Leiden clusters:', adata_work.obs['leiden_scvi'].nunique())


## 8. Side-by-side comparison: Harmony vs scVI

In [ ]:
# 2x2 grid: rows = method, cols = donor / lineage
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

sc.pl.embedding(adata_work, basis='X_umap_harmony', color='donor_id',
                ax=axes[0, 0], show=False, frameon=False,
                title='Harmony — donor', legend_loc='on data', legend_fontsize=6)
sc.pl.embedding(adata_work, basis='X_umap_harmony', color='ClusterTop',
                ax=axes[0, 1], show=False, frameon=False,
                title='Harmony — lineage (author labels)')

sc.pl.embedding(adata_work, basis='X_umap_scvi', color='donor_id',
                ax=axes[1, 0], show=False, frameon=False,
                title='scVI — donor', legend_loc='on data', legend_fontsize=6)
sc.pl.embedding(adata_work, basis='X_umap_scvi', color='ClusterTop',
                ax=axes[1, 1], show=False, frameon=False,
                title='scVI — lineage (author labels)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'hero_harmony_vs_scvi.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / 'hero_harmony_vs_scvi.tiff', dpi=300, bbox_inches='tight',
            pil_kwargs={'compression': 'tiff_lzw'})
plt.show()


## 9. Integration quality metric: per-cell donor mixing entropy

For each cell, look at its k=50 nearest neighbors in the embedding and compute the Shannon entropy
of their donor labels. Higher = better mixing across donors (less batch residual). We also report
mean silhouette of `ClusterTop` (author lineage labels) as a "biology preservation" proxy.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.stats import entropy
from sklearn.metrics import silhouette_score

def donor_mixing_entropy(emb, donors, k=50, sample_size=5000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(emb.shape[0], size=min(sample_size, emb.shape[0]), replace=False)
    nn = NearestNeighbors(n_neighbors=k).fit(emb)
    _, ind = nn.kneighbors(emb[idx])
    donors_arr = np.asarray(donors)
    ents = []
    for row in ind:
        _, counts = np.unique(donors_arr[row], return_counts=True)
        ents.append(entropy(counts, base=2))
    return float(np.mean(ents)), float(np.std(ents))

def biology_silhouette(emb, labels, sample_size=5000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(emb.shape[0], size=min(sample_size, emb.shape[0]), replace=False)
    return float(silhouette_score(emb[idx], np.asarray(labels)[idx]))

donors = adata_work.obs['donor_id'].astype(str).values
lineage = adata_work.obs['ClusterTop'].astype(str).values

results = {}
for name, key in [('uncorrected', 'X_pca'),
                  ('harmony',     'X_pca_harmony'),
                  ('scvi',        'X_scvi')]:
    emb = adata_work.obsm[key]
    me, sd = donor_mixing_entropy(emb, donors, k=50)
    sil = biology_silhouette(emb, lineage)
    results[name] = {'mixing_entropy': me, 'mixing_sd': sd, 'lineage_silhouette': sil}
    print(f'{name:12s}  mixing_entropy={me:.3f} ± {sd:.3f}   lineage_silhouette={sil:.3f}')

metrics_df = pd.DataFrame(results).T
metrics_df.to_csv(RES_DIR / 'integration_metrics.csv')
metrics_df


In [ ]:
# Visualize the trade-off
fig, ax = plt.subplots(figsize=(7, 5))
for name, row in metrics_df.iterrows():
    ax.scatter(row['mixing_entropy'], row['lineage_silhouette'], s=200, label=name)
    ax.annotate(name, (row['mixing_entropy'], row['lineage_silhouette']),
                xytext=(8, 5), textcoords='offset points', fontsize=11)
ax.set_xlabel('Donor mixing entropy (higher = better batch correction)')
ax.set_ylabel('Lineage silhouette (higher = biology preserved)')
ax.set_title('Integration trade-off')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'integration_tradeoff.png', dpi=200, bbox_inches='tight')
plt.show()


## 10. MMRd vs MMRp compositional comparison

In [ ]:
# Lineage composition per MMR status
comp = (adata_work.obs.groupby(['MMRStatusTumor', 'ClusterTop'], observed=True)
        .size().rename('n').reset_index())
comp_wide = comp.pivot(index='MMRStatusTumor', columns='ClusterTop', values='n').fillna(0)
comp_frac = comp_wide.div(comp_wide.sum(axis=1), axis=0)
print(comp_frac.round(3))
comp_frac.to_csv(RES_DIR / 'mmr_lineage_fractions.csv')

fig, ax = plt.subplots(figsize=(9, 4))
comp_frac.plot(kind='barh', stacked=True, ax=ax, colormap='tab10', width=0.7)
ax.set_xlabel('Fraction of cells')
ax.set_ylabel('')
ax.set_title('Lineage composition: MMRd vs MMRp (tumor only)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'mmr_lineage_composition.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Per-donor composition heatmap (so reader sees within-group variance)
donor_comp = (adata_work.obs.groupby(['donor_id', 'ClusterTop'], observed=True)
              .size().unstack(fill_value=0))
donor_comp_frac = donor_comp.div(donor_comp.sum(axis=1), axis=0)

donor_mmr_map = (adata_work.obs.groupby('donor_id', observed=True)['MMRStatusTumor']
                 .agg(lambda s: s.mode()[0]))
donor_comp_frac = donor_comp_frac.loc[donor_mmr_map.sort_values().index]
row_colors = donor_mmr_map.loc[donor_comp_frac.index].map({'MMRd': '#d62728', 'MMRp': '#1f77b4'})

g = sns.clustermap(donor_comp_frac, row_colors=row_colors, row_cluster=False,
                   col_cluster=False, cmap='viridis', figsize=(8, 5),
                   cbar_kws={'label': 'lineage fraction'})
g.ax_heatmap.set_xlabel('Lineage')
g.ax_heatmap.set_ylabel('Donor')
plt.savefig(FIG_DIR / 'mmr_donor_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()


## 11. Save artifacts

In [ ]:
# Save subsetted + integrated AnnData (without raw counts to keep file small)
out = adata_work.copy()
# Drop the heavy counts layer for the saved file (we have it on disk via the original)
if 'counts' in out.layers:
    del out.layers['counts']
out.write_h5ad(RES_DIR / 'pelka_subset_integrated.h5ad', compression='gzip')
print('Saved:', RES_DIR / 'pelka_subset_integrated.h5ad')

# Manifest
manifest = {
    'dataset': 'Pelka 2021 (CELLxGENE collection 4d82bd8e-8827-410b-8fc7-685b7dd92585)',
    'donors_selected_MMRd': mmrd_donors,
    'donors_selected_MMRp': mmrp_donors,
    'n_cells_after_qc': int(adata_work.n_obs),
    'n_genes': int(adata_work.n_vars),
    'n_hvgs': int(adata_work.var['highly_variable'].sum()),
    'harmony_clusters': int(adata_work.obs['leiden_harmony'].nunique()),
    'scvi_clusters': int(adata_work.obs['leiden_scvi'].nunique()),
    'integration_metrics': results,
}
with open(RES_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('Wrote manifest.json')


---
## Summary

- **Subset:** 8 donors (4 MMRd + 4 MMRp), tumor only, from Pelka 2021 CRC atlas.
- **Pipeline:** QC -> log-norm -> batch-aware HVG (3000) -> PCA -> {Harmony, scVI} -> Leiden, UMAP.
- **Metrics:** Donor mixing entropy (batch removal) and lineage silhouette (biology preservation).
- **Output:** Hero 2x2 (Harmony vs scVI x donor vs lineage), trade-off scatter, MMR composition.

See `results/integration_metrics.csv` for the numeric comparison and `results/manifest.json` for run config.
